# Chapter 3 — Continuum normalisation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/the-cannon-tutorial/blob/main/notebooks/03_continuum.ipynb)

**Goal.** The Cannon models *continuum-normalised* spectra: the flux divided by an estimate of the stellar
continuum, so that only the absorption lines remain. Here we build that normalisation following Ness et al. 2015
(N15, §2.3 and §5.3), and test the one property The Cannon needs from it: that it does not change with the S/N of
the spectrum. A model trained on S/N = 300 spectra and applied to S/N = 30 spectra is only right if both are
normalised in the same way.

**You will learn**
1. the simplest pseudo-continuum, an upper quantile of the flux in a running window, and why it depends on S/N;
2. how to find **continuum pixels** — pixels whose normalised flux is ≈ 1 and does not depend on the labels —
   from a first Cannon model;
3. to fit a low-order polynomial to those pixels only, with inverse-variance weights, per detector;
4. to measure the S/N dependence of a normalisation by adding noise to high-S/N spectra.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are extracts of APOGEE DR17 and Gaia DR3, published on Zenodo (https://doi.org/10.5281/zenodo.23813051).
# MANIFEST.json lists, for each file, the query that made it and its SHA-256 checksum.
DATA_URL = 'https://zenodo.org/records/23813051/files/{}?download=1'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL.format('MANIFEST.json')).read())


def load(name, cache=Path('data')):
    """Download <name> (.parquet table or .npz arrays) from Zenodo if needed, check its SHA-256, read it.

    A .parquet file is returned as a pandas DataFrame, an .npz file as a dict of numpy arrays.
    """
    ext = '.npz' if name.startswith(('spectra_', 'rvs_')) else '.parquet'
    path = cache / f'{name}{ext}'
    sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
    if not path.exists() or sha(path) != MANIFEST[name]['sha256']:   # first use, or a stale/damaged copy
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        part = path.with_suffix('.part')
        urllib.request.urlretrieve(DATA_URL.format(path.name), part)   # download to a temporary file first
        if sha(part) != MANIFEST[name]['sha256']:
            raise IOError(f'{path.name}: checksum does not match the manifest (incomplete download?)')
        part.replace(path)                                         # only a verified file takes the final name
    # the checksum guarantees that everyone runs this chapter on exactly the same data
    if ext == '.parquet':
        return pd.read_parquet(path)
    with np.load(path, allow_pickle=False) as f:   # allow_pickle=False: arrays only, no code can be loaded
        return {k: f[k] for k in f.files}
# --- Spectra and continuum normalisation (as in Chapters 0 and 3) ----------------------------------------
NPIX = 8575
wave = 10**(4.179 + 6e-6 * np.arange(NPIX))               # apStar vacuum wavelength grid [Angstrom]
CHIPS = [(15150, 15800), (15865, 16425), (16480, 16950)]  # the three detectors [A], inside the edges of Chapter 0
BAD_BITS = sum(1 << b for b in [0, 1, 2, 3, 4, 5, 6, 7, 12])   # pixels given zero weight (Chapter 0)


def prepare(sp, rows=slice(None)):
    """flux and err (float64) of the spectra file sp; err = inf where there are no data, a BAD_BITS flag, or
    outside CHIPS (the detector edges, where only some spectra have data)."""
    flux = sp['flux'][rows].astype(float)
    err = sp['err'][rows].astype(float)
    on_chip = np.any([(wave > lo) & (wave < hi) for lo, hi in CHIPS], axis=0)   # pixels at the detector edges and
    bad = ((sp['mask'][rows] & BAD_BITS) > 0) | ~np.isfinite(flux) | ~np.isfinite(err) | (err <= 0) | ~on_chip  # gaps: no weight
    return np.where(bad, 0.0, flux), np.where(bad, np.inf, err)


def cheb_normalise(flux, err, cont, order=2):
    """Divide by a Chebyshev polynomial of the given order fitted, per detector, to the continuum pixels `cont`
    (boolean, NPIX) with weights 1/err^2 (N15 sect. 2.3 and 5.3). Returns normalised flux and error;
    pixels outside the detectors get err = inf."""
    fn, en = np.zeros_like(flux), np.full_like(err, np.inf)
    for lo, hi in CHIPS:
        c = (wave > lo) & (wave < hi)
        t = (wave[c] - (lo + hi) / 2) / ((hi - lo) / 2)          # wavelength mapped to [-1, 1]
        B = np.polynomial.chebyshev.chebvander(t, order)         # (pixels, order+1) basis
        use = cont[c]
        w = 1 / err[:, c][:, use]**2                             # (N, continuum pixels); 0 for flagged pixels
        # one small weighted least-squares problem per star: (B^T W B) a = B^T W f
        BtWB = np.einsum('pi,np,pj->nij', B[use], w, B[use])
        BtWf = np.einsum('pi,np->ni', B[use], w * flux[:, c][:, use])
        a = np.linalg.solve(BtWB, BtWf[:, :, None])[:, :, 0]     # (N, order+1); b as a stack of column vectors
        cont_fit = a @ B.T                                       # (N, pixels of this detector)
        fn[:, c] = flux[:, c] / cont_fit
        en[:, c] = err[:, c] / cont_fit
    return fn, en
# --- The Cannon (as written in Chapter 2) -----------------------------------------------------------------
from scipy.optimize import least_squares


def vectorize(x):
    """Quadratic label vector v(x) = [1, x_k, x_k x_l (k <= l)] for scaled labels x of shape (N, K): (N, D)."""
    x = np.atleast_2d(x)
    k, l = np.triu_indices(x.shape[1])
    return np.hstack([np.ones((len(x), 1)), x, x[:, k] * x[:, l]])


def dvectorize(x):
    """Derivative dv/dx of the label vector of one star (x of shape (K,)): (D, K)."""
    K = len(x)
    k, l = np.triu_indices(K)
    J = np.zeros((1 + K + len(k), K))
    J[1:1 + K] = np.eye(K)
    for i, (a, b) in enumerate(zip(k, l)):
        J[1 + K + i, a] += x[b]
        J[1 + K + i, b] += x[a]
    return J


def fit_theta(V, flux, var):
    """Weighted least squares at every pixel: theta (P, D) and its covariance (P, D, D); weights 1/var (N, P).
    Zero-weight entries (var = inf) drop out of the sums."""
    w = 1 / var
    flux = np.where(w > 0, flux, 0.0)                       # NaN flux at zero weight must not poison the sums
    N, Dv = V.shape
    A = ((V[:, :, None] * V[:, None, :]).reshape(N, Dv * Dv).T @ w).T.reshape(-1, Dv, Dv)
    b = (V.T @ (w * flux)).T
    cov = np.linalg.pinv(A)                                 # pinv: pixels without data give theta = 0, not an error
    return np.einsum('pde,pe->pd', cov, b), cov


S_GRID = np.r_[0, np.logspace(-4, 0, 41)]                  # 0, then 1e-4 ... 1 in steps of 0.1 dex


def fit_scatter(resid, err2):
    """Maximum-likelihood scatter s per pixel for fixed residuals: best value on the grid S_GRID, refined by a
    parabola in log s (Chapter 2). Zero-weight entries (err2 = inf) are ignored."""
    use = np.isfinite(err2) & np.isfinite(resid)
    r2 = np.where(use, resid**2, 0.0)
    e2 = np.where(use, err2, 1.0)
    nll = np.array([np.sum(use * (r2 / (e2 + s**2) + np.log(e2 + s**2)), axis=0) for s in S_GRID])
    best = np.argmin(nll, axis=0)
    j = np.clip(best, 2, len(S_GRID) - 2)
    p = np.arange(nll.shape[1])
    a, b, c = nll[j - 1, p], nll[j, p], nll[j + 1, p]
    shift = np.clip(0.5 * (a - c) / np.where(a - 2 * b + c > 0, a - 2 * b + c, np.inf), -1, 1)   # stay near the grid optimum
    return np.where(best == 0, 0.0, S_GRID[j] * np.exp(shift * np.log(S_GRID[2] / S_GRID[1])))


def train(x, flux, err, n_iter=3):
    """Training step. x: scaled labels (N, K); flux, err: (N, P) (err = inf for pixels to ignore).
    Returns theta (P, D), its covariance (P, D, D) and the scatter s (P,)."""
    V, err2 = vectorize(x), err**2
    s = np.zeros(flux.shape[1])
    for _ in range(n_iter):
        theta, cov = fit_theta(V, flux, err2 + s**2)
        s = fit_scatter(flux - V @ theta.T, err2)
    theta, cov = fit_theta(V, flux, err2 + s**2)
    return theta, cov, s


def test(flux, err, theta, s, x0=None):
    """Test step for one spectrum: scaled labels (K,), covariance (K, K), chi^2, number of pixels used, converged?"""
    K = int(round((np.sqrt(8 * theta.shape[1] + 1) - 3) / 2))   # invert D = 1 + K + K(K+1)/2
    ok = np.isfinite(flux) & np.isfinite(err) & (theta[:, 0] != 0)
    w = 1 / np.sqrt(err[ok]**2 + s[ok]**2)
    th, f = theta[ok], flux[ok]
    res = lambda x: (f - th @ vectorize(x)[0]) * w
    jac = lambda x: -(th @ dvectorize(x)) * w[:, None]
    r = least_squares(res, np.zeros(K) if x0 is None else x0, jac=jac, method='lm')
    # covariance from the Jacobian at the optimum (pinv: no crash if a label is unconstrained); r.success: converged
    return r.x, np.linalg.pinv(r.jac.T @ r.jac), np.sum(r.fun**2), ok.sum(), r.success


def train_clean(x, flux, err, cut=3.0):
    """Train, drop the training stars whose mean squared residual per pixel (in units of sigma^2 + s^2) exceeds
    `cut`, and train again (Chapter 4). Returns theta, its covariance, s, and the boolean mask of stars kept."""
    theta, cov, s = train(x, flux, err)
    z2 = (flux - vectorize(x) @ theta.T)**2 / (err**2 + s**2)
    z2 = np.where(np.isfinite(err) & (theta[:, 0] != 0), z2, np.nan)
    keep = np.nanmean(z2, axis=1) < cut
    theta, cov, s = train(x[keep], flux[keep], err[keep])
    return theta, cov, s, keep


def scatter_stats(d):
    """Summary of differences d (e.g. Cannon - reference): mean (bias), standard deviation, root-mean-square error,
    robust standard deviation 1.4826 x median absolute deviation (= std for a Gaussian, insensitive to outliers),
    and the fraction of points more than 3 robust std from the median (the tails)."""
    d = np.asarray(d)[np.isfinite(d)]
    med = np.median(d)
    rob = 1.4826 * np.median(np.abs(d - med))
    return {'bias': d.mean(), 'std': d.std(), 'rmse': np.sqrt(np.mean(d**2)), 'robust': rob,
            'tail': np.mean(np.abs(d - med) > 3 * rob)}

labels = load('labels')
tr = labels[labels['sample'] == 'train'].reset_index(drop=True)
spec = load('spectra_train')
lines = load('linelist')
flux, err = prepare(spec)                                  # err = inf at flagged pixels and outside the detectors
L = tr[['teff', 'logg', 'fe_h']].to_numpy()
print(f'{len(tr)} training spectra, {NPIX} pixels each')

## 3.1 A running-quantile pseudo-continuum

Absorption lines only remove flux, so the continuum lies near the *top* of the spectrum. N15 started from the
90th percentile of the flux in a window of 50 Å moved along the spectrum. We compute it on a coarse grid of
window centres (every 10 pixels) within each detector and interpolate in between.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view


def quantile_continuum(flux, err, q=90, width=50.0, step=10):
    """q-th percentile of the good pixels in a running window of `width` A, per detector (N15 sect. 2.3).
    Returns the continuum (N, NPIX); NaN outside the detectors."""
    cont = np.full(flux.shape, np.nan)
    for lo, hi in CHIPS:
        c = np.flatnonzero((wave > lo) & (wave < hi))
        npw = int(width / (np.median(wave[c]) * np.log(10) * 6e-6))      # window length in pixels (~230)
        f = np.where(np.isfinite(err[:, c]), flux[:, c], np.nan)          # flagged pixels do not vote
        win = sliding_window_view(f, npw, axis=1)[:, ::step]               # (N, centres, npw): a view, no copy
        qc = np.nanpercentile(win, q, axis=2)                              # (N, centres)
        centre = np.arange(win.shape[1]) * step + npw // 2                 # pixel index (within c) of each centre
        for n in range(len(f)):
            cont[n, c] = np.interp(np.arange(len(c)), centre, qc[n])       # constant beyond the first/last centre
    return cont


cq = quantile_continuum(flux, err)
fq, eq = flux / cq, err / cq                                  # quantile-normalised flux and error
i = np.argmin(np.abs(tr.teff - 4500) + 300 * np.abs(tr.logg - 2.0) + 1000 * np.abs(tr.fe_h))
fig, ax = plt.subplots(2, 1, figsize=(14, 6.5), sharex=True)
good = np.isfinite(err[i])
show = lambda f: np.where(good, f, np.nan)                     # NaN breaks the line at gaps and flagged pixels
ax[0].plot(wave, show(flux[i]), 'k', lw=0.4, label='flux / median')
ax[0].plot(wave, cq[i], 'C1', lw=1.5, label='running 90th percentile (50 Å)')
ax[0].set(ylabel='flux', ylim=(0.3, 1.5))
ax[0].legend(fontsize=8)
ax[1].plot(wave, show(fq[i]), 'k', lw=0.4)
ax[1].axhline(1, color='C1', lw=1)
ax[1].set(xlabel='vacuum wavelength [Å]', ylabel='normalised flux', ylim=(0.3, 1.2))
plt.tight_layout()

**Figure 3.1.** Top: the spectrum of a typical training giant (black; flagged pixels omitted) and its running
90th-percentile continuum (orange). Bottom: the ratio, a pseudo-continuum-normalised spectrum (the few upward
spikes are sky-line residuals that the mask does not flag). The method follows
the broad shape of the spectrum well, but by construction 10% of the pixels in every window lie above 1, and the
level it assigns depends on how many lines are in the window — and on the noise, as the next test shows.

## 3.2 The S/N test

To see how a normalisation responds to noise, take high-S/N spectra, add Gaussian noise to bring them to a lower
S/N, normalise both versions, and compare. If a spectrum has errors $\sigma$ and we want errors
$\sigma' = f/{\rm (S/N)}' > \sigma$, we add noise of variance $\sigma'^2 - \sigma^2$ (the noise already there
counts). The **bias** of the normalisation is the mean difference between the normalised noisy spectrum and the
normalised original, at pixels that are continuum in the original. We average over 200 stars, so the random part
of the difference averages away.

In [ ]:
rng = np.random.default_rng(3)
sub = rng.choice(len(tr), 200, replace=False)                 # 200 training stars for the test


def degrade(f, e, snr):
    """Add Gaussian noise so that the per-pixel S/N becomes snr (where it was higher)."""
    target = np.abs(f) / snr
    extra = np.sqrt(np.clip(target**2 - e**2, 0, None))
    e2 = np.where(np.isfinite(e), np.maximum(e, target), np.inf)
    return f + rng.normal(size=f.shape) * np.where(np.isfinite(e), extra, 0), e2


SNRS = [20, 30, 50, 100, 200]
f0, e0 = flux[sub], err[sub]
ref_q = f0 / quantile_continuum(f0, e0)
bias_q = []
for snr in SNRS:
    fd, ed = degrade(f0, e0, snr)
    nq = fd / quantile_continuum(fd, ed)
    sel = np.isfinite(e0) & (ref_q > 0.98)                    # pixels near the continuum in the original
    bias_q.append(np.mean((nq - ref_q)[sel]))
print('quantile normalisation, mean change of the continuum level vs S/N:', dict(zip(SNRS, np.round(bias_q, 4))))

At S/N = 30 the normalised continuum drops by ≈ 2% (≈ 4% at S/N = 20): noise scatters pixels upwards as well as
downwards, so the 90th percentile of a noisy window lies higher than that of the noiseless one, and dividing by it
pushes the whole spectrum down. The change of a continuum level is small, but it is
systematic and S/N-dependent — exactly what makes a model trained at high S/N biased at low S/N (N15 §2.3).

## 3.3 Continuum pixels from a first model

N15 (§2.3, §5.3) define the continuum *with the model itself*: train a Cannon model on the
quantile-normalised spectra, and call continuum the pixels where the model's baseline spectrum $\theta_0$ is
close to 1 **and** the flux hardly depends on any label. Those pixels are fixed once and for all; every spectrum —
training or test, high or low S/N — is then normalised with the same pixels.

We train a three-label model ($T_{\rm eff}$, $\log g$, [Fe/H]) on the quantile-normalised training spectra and
apply N15's criteria: $|\theta_0 - 1| < 0.015$ and first-order coefficients (per unit of the *unscaled* labels)
below $10^{-5}$ per K, 0.0045 per dex of $\log g$ and 0.0085 per dex of [Fe/H].

In [ ]:
pivot = np.median(L, axis=0)
scale = (np.percentile(L, 97.5, axis=0) - np.percentile(L, 2.5, axis=0)) / 2
x = (L - pivot) / scale
eq_use = np.where(np.isfinite(fq), eq, np.inf)
theta_q, _, s_q = train(x, np.nan_to_num(fq), eq_use)
dphys = theta_q[:, 1:4] / scale                                # coefficients per K, per dex, per dex
LIMITS = np.array([1e-5, 0.0045, 0.0085])                      # N15 sect. 5.3
on_chip = np.isfinite(cq[0])
cont = on_chip & (np.abs(theta_q[:, 0] - 1) < 0.015) & np.all(np.abs(dphys) < LIMITS, axis=1)
print(f'continuum pixels: {cont.sum()} of {on_chip.sum()} pixels on the detectors ({cont.sum() / on_chip.sum():.1%}); '
      'per detector:', [int(cont[(wave > lo) & (wave < hi)].sum()) for lo, hi in CHIPS])

fig, ax = plt.subplots(3, 1, figsize=(14, 8))
for a, (lo, hi) in zip(ax, [(15200, 15330), (16150, 16280), (16700, 16830)]):
    z = (wave > lo) & (wave < hi)
    a.plot(wave[z], theta_q[z, 0], 'k', lw=1, label=r'baseline $\theta_0$')
    a.plot(wave[z & cont], theta_q[z & cont, 0], 'o', color='C2', ms=4, label='continuum pixels')
    for d, nm in zip(range(3), [r'$T_{\rm eff}$ (per 1000 K)', r'$\log g$ (per dex)', '[Fe/H] (per dex)']):
        a.plot(wave[z], 0.75 + dphys[z, d] * (1000 if d == 0 else 1), lw=0.7, label=r'0.75 + $\partial f/\partial$' + nm)
    a.set(ylabel='normalised flux', ylim=(0.55, 1.08))
ax[0].legend(fontsize=7, ncol=5, loc='lower left')
ax[-1].set(xlabel='vacuum wavelength [Å]')
plt.tight_layout()

**Figure 3.2.** The first model, trained on quantile-normalised spectra, in three 130 Å windows (one per
detector). Black: the baseline spectrum $\theta_0$ (a giant with the median training labels). Coloured lines,
offset to 0.75: the derivatives of the flux with respect to $T_{\rm eff}$ (per 1000 K), $\log g$ and [Fe/H] (per
dex). Green dots: the continuum pixels — near 1 in $\theta_0$ and with all three derivatives small. They sit
between the lines, a few per window.

Are they really free of lines? Compare their distance to the nearest line of the line list with that of all
pixels on the detectors.

In [ ]:
strong = lines[lines.strength > lines.groupby('species').strength.transform('max') - 1.0]   # within 1 dex of each species' strongest
dist = np.min(np.abs(wave[:, None] - strong.wave_vac.to_numpy()[None, :]), axis=1)          # A to the nearest strong line
near_all, near_cont = np.mean(dist[on_chip] < 0.5), np.mean(dist[cont] < 0.5)
print(f'fraction within 0.5 A of a strong listed line: all detector pixels {near_all:.3f}, continuum pixels {near_cont:.3f}')
# Test: continuum pixels avoid strong lines (at most half as often within 0.5 A of one as an average pixel).
assert near_cont < 0.5 * near_all

## 3.4 Normalising with the continuum pixels

On each detector, fit a second-order Chebyshev polynomial in wavelength to the flux of the continuum pixels only,
weighting each pixel by $1/\sigma^2$ (flagged pixels have zero weight), and divide the spectrum by it (N15 §2.3).
This is a small weighted least-squares fit per star and detector; `cheb_normalise` (defined at the top) does it
for all stars at once.

The pixel list we derived is also shipped with the tutorial as the file `continuum_pixels` (computed by
`scripts/build_data.py` with the same code), so that later chapters do not have to repeat this chapter. Check
that the two agree:

In [ ]:
shipped = load('continuum_pixels')['continuum'].to_numpy()
print(f'shipped list: {shipped.sum()} pixels; identical to ours: {np.array_equal(shipped, cont)}')
assert np.array_equal(shipped, cont)

fc, ec = cheb_normalise(flux, err, cont)
fig, ax = plt.subplots(figsize=(14, 4))
z = (wave > 15600) & (wave < 15800) & np.isfinite(err[i])
ax.plot(wave[z], fq[i, z], color='C1', lw=0.8, label='running 90th percentile')
ax.plot(wave[z], fc[i, z], 'k', lw=0.8, label='Chebyshev on continuum pixels')
ax.plot(wave[z & cont], fc[i, z & cont], 'o', color='C2', ms=4, label='continuum pixels')
ax.axhline(1, color='0.5', lw=0.5)
ax.set(xlabel='vacuum wavelength [Å]', ylabel='normalised flux')
ax.legend(fontsize=8)
plt.tight_layout()
with np.errstate(invalid='ignore', divide='ignore'):
    print(f'star {i}: median ratio of the two normalisations = {np.nanmedian(fc[i, on_chip] / fq[i, on_chip]):.4f}')

**Figure 3.3.** A 200 Å section of the typical giant normalised in the two ways: running 90th percentile
(orange) and second-order Chebyshev polynomials fitted to the continuum pixels (black; green dots). The two
differ by a slowly varying factor of a few tenths of a per cent (median ratio 1.004 for this star): the running
quantile puts the "continuum" near the top of the noise and of the weak lines, the polynomial through the
continuum pixels near their mean. Neither is the true
continuum of the star; The Cannon only needs the *same* definition for every spectrum.

## 3.5 The S/N test again

Now repeat the noise-injection test of §3.2 with the continuum-pixel normalisation. The division by the fitted
continuum is not exactly linear in the noise (a ratio of noisy quantities), so a small bias can remain: measure
it, do not assume it is zero.

In [ ]:
ref_c = cheb_normalise(f0, e0, cont)[0]
bias_c = []
for snr in SNRS:
    fd, ed = degrade(f0, e0, snr)
    nc = cheb_normalise(fd, ed, cont)[0]
    sel = np.isfinite(e0) & (ref_c > 0.98) & np.isfinite(nc)
    bias_c.append(np.mean((nc - ref_c)[sel]))
print('continuum-pixel normalisation, mean change vs S/N:', dict(zip(SNRS, np.round(bias_c, 5))))
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(SNRS, np.abs(bias_q) * 100, 'o-', label='running 90th percentile')
ax.plot(SNRS, np.abs(bias_c) * 100, 's-', label='Chebyshev on continuum pixels')
ax.set(xscale='log', yscale='log', xlabel='S/N after adding noise', ylabel='|change of continuum level| [%]')
ax.legend(fontsize=8)
plt.tight_layout()
k30 = SNRS.index(30)
# Test: at S/N = 30 the quantile normalisation shifts the continuum at least 3 times more than the fixed-pixel one.
assert abs(bias_q[k30]) > 3 * abs(bias_c[k30])

**Figure 3.4.** Mean change of the normalised flux at continuum pixels when 200 high-S/N training spectra are
degraded to a lower S/N (horizontal axis) and normalised again, for the two methods (absolute values, log
scales). The running quantile moves the continuum by ≈ 1% at S/N = 50, 2% at 30 and 4% at 20. The
continuum-pixel fit changes it by less than $10^{-4}$ at every S/N — consistent with zero at the precision of this
test — because the same pixels, with the same weights, define the continuum at any S/N.

## Summary

- Quantile-based pseudo-continua depend on S/N; that would bias any model trained at high S/N and used at low S/N.
- Continuum pixels are chosen from a first Cannon model: baseline flux ≈ 1 and no dependence on the labels
  (N15's criteria select ≈ 9% of the pixels here; N15 had 6.5%).
- A low-order polynomial fitted to those pixels, per detector, with inverse-variance weights, is a normalisation
  whose S/N dependence we measured to be small.
- From now on every spectrum is normalised this way, with the shipped list `continuum_pixels`.

**Exercises.**
1. Use a 4th-order instead of a 2nd-order Chebyshev polynomial. Does the S/N bias change? Does the normalised
   flux at the detector edges?
2. Choose the continuum pixels at random (same number). Casey et al. 2016 argue that any fixed set of pixels would
   work for The Cannon; what do you lose?

## References

- Casey et al. 2016, *The Cannon 2: A data-driven model of stellar spectra for detailed chemical abundance analyses*, arXiv:1603.03040. [ADS](https://ui.adsabs.harvard.edu/abs/2016arXiv160303040C)
- Ness et al. 2015, *The Cannon: A data-driven approach to Stellar Label Determination*, ApJ 808, 16. [ADS](https://ui.adsabs.harvard.edu/abs/2015ApJ...808...16N)
- Ness et al. 2018, *Galactic Doppelgängers: The Chemical Similarity Among Field Stars and Among Stars with a Common Birth Origin*, ApJ 853, 198. [ADS](https://ui.adsabs.harvard.edu/abs/2018ApJ...853..198N)
- Santos-Peral et al. 2020, *The AMBRE Project: Spectrum normalisation influence on Mg abundances in the metal-rich Galactic disc*, A&A 639, A140. [ADS](https://ui.adsabs.harvard.edu/abs/2020A%26A...639A.140S)